# Waste as a Measured Input: A Causal Network Model of Nutrient Recycling with Growth-Modulated Transfer in Integrated Insect–Livestock–Fish–Crop Systems

**Firstname Lastname** (email@example.org), **Second Author** (email2@example.org) — September 2026

> **Abstract.** In integrated food-production systems, the waste of one organism is a primary feed input for another, so the nutritional state of the whole community is a closed-loop function of itself. We present a causal network model in which every inter-organism flow is a measurable nutrient vector, every edge weight is a digestibility-conditioned transfer efficiency, and waste-derived feed modulates the growth rate of the consumer through explicit penalty and complementarity functions. The model admits (i) a waste quality index, (ii) a total causal effect matrix given by a nutrient-wise Leontief inverse, and (iii) critical waste fractions at which growth no longer covers maintenance. An illustrative six-node system (plants–crickets–quail–fish–black soldier fly larvae–soil pool) shows that 34–60% of steady-state protein output arrives through indirect recycling pathways, that quail growth turns negative beyond a waste fraction of φ* ≈ 0.41, and that the detritivore node supports an interior optimum at φ ≈ 0.04 with break-even at φ ≈ 0.71.

*This notebook is the executable version of the paper: every figure and number in Section 4 is regenerated by the code cells below. Requirements: `numpy`, `matplotlib`.*

## 1. Introduction

Closed-loop food systems couple plants, crickets, poultry, fish, and black soldier fly larvae (BSFL). The modeling bottleneck is conceptual: **waste is almost universally treated as a loss term** — wrong twice over in a circular system. Waste (1) carries a measurable nutrient vector (it is an asset), and (2) is a function of feed while feed is partly a function of waste (nutrition is an output of the *network*, not of any single feeding event).

Contributions: **(1)** waste as a measured flow (a waste quality index, Section 3.6); **(2)** growth-modulated transfer: break-even ($\phi^*$) and interior-optimum ($\phi^{opt}$) waste fractions (Section 3.3, 4.3); **(3)** total causal effects via a nutrient Leontief inverse and a probabilistic culling governor (Sections 3.5, 3.9).

## 2. System description

Plants partition output among crickets, quail, fish, BSFL and the soil pool; crickets feed quail and supply the BSFL farm; quail and fish send their waste streams to BSFL; larvae return to quail and fish; frass closes the soil loop. External inputs enter at the plant node; export is drawn only after internal requirements and reserves are satisfied.

In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle

nodes=['Plants','Crickets','Quail','Fish','BSFL','Soil pool']
C=np.array([[0,0.22,0.16,0.14,0.10,0.16],[0,0,0.45,0,0.35,0],[0,0,0,0,0.45,0],
            [0,0,0,0,0.35,0],[0,0,0.28,0.42,0,0.35],[0.95,0,0,0,0,0]])   # routing fractions
eta={(0,1):[.50,.55,.40,.10],(0,2):[.55,.50,.45,.08],(0,3):[.50,.50,.40,.05],
     (0,4):[.40,.45,.30,.05],(0,5):[.05,.35,.10,.60],(1,2):[.60,.65,.55,.15],
     (1,4):[.45,.50,.35,.10],(2,4):[.40,.45,.50,.20],(3,4):[.40,.45,.40,.15],
     (4,2):[.65,.70,.70,.30],(4,3):[.65,.70,.70,.30],(5,0):[.25,.60,.20,.80]}
pos={'Plants':(.5,.85),'Crickets':(.15,.55),'Quail':(.42,.42),'Fish':(.78,.55),'BSFL':(.62,.18),'Soil pool':(.16,.12)}
fig,ax=plt.subplots(figsize=(8,6)); ax.axis('off'); ax.set_xlim(0,1); ax.set_ylim(0,1)
for (i,j) in eta:
    if C[i,j]==0: continue
    ax.add_patch(FancyArrowPatch(pos[nodes[i]],pos[nodes[j]],connectionstyle='arc3,rad=0.12',
        arrowstyle='-|>',mutation_scale=14,lw=1.5+5*C[i,j],color='#2b6cb0',alpha=.85))
for n,(x,y) in pos.items():
    ax.add_patch(Circle((x,y),.055,color='#c6f6d5',ec='#22543d',lw=1.6,zorder=5))
    ax.text(x,y,n,ha='center',va='center',fontsize=9,fontweight='bold',zorder=6)
ax.set_title('Figure 1 — Integrated production network with waste-as-feed recycling'); plt.show()

## 3. Model

### 3.1–3.2 Nutrient vector, mass balance, and digestibility-conditioned waste

Let $\mathbf n_i\in\mathbb{R}^q_{\ge0}$ be the nutrient composition of node $i$'s output (ME, CP, EE, Ca, …). For consumer $j$:

$$\mathbf n^{\mathrm{feed}}_j=\mathbf n^{\mathrm{retained}}_j+\mathbf n^{\mathrm{waste}}_j+\mathbf n^{\mathrm{product}}_j$$

$$\mathbf w_{i\to j}(t)=\mathbf f_i(t)\odot(\mathbf 1-\mathbf d_{ij}(t)),\qquad \mathbf p_{i\to j}(t)=\mathbf f_i(t)\odot\mathbf d_{ij}(t)\odot\mathbf r_j$$

$$\mathbf f_j(t+1)=\sum_i[\mathbf p_{i\to j}(t)+\mathbf w_{i\to j}(t)]+\mathbf e_j(t+1)$$

Waste quality is measurable because digestibility $\mathbf d_{ij}$ is measurable (in-vitro digestion or feeding trials).

In [ ]:
# --- 3.2 demo: feed composition -> waste composition via digestibility ---
nutrients=['ME(kcal)','CP(g)','EE(g)','Ca(g)']
feed=np.array([1500,110,15,4.])                       # 1 kg plant-mix biomass
d_plant_to_cricket=np.array([.50,.55,.40,.10])        # digestibility vector
waste=feed*(1-d_plant_to_cricket)
print('feed :',dict(zip(nutrients,feed)))
print('waste:',dict(zip(nutrients,np.round(waste,2))), ' <- becomes BSFL feed')

### 3.3 Waste-modulated growth

$$\mu_j(t)=\mu_j^{\max} f_E f_P f_W f_T f_\rho,\qquad d_j(t)=\frac{d_j^{\max}}{1+\beta_d CF_j(t)+\gamma_d\phi_j(t)},\qquad f_W=1-\alpha_W\phi_j g_W(\mathbf w_j)$$

with waste-derived feed fraction $\phi_j$. For detritivores (BSFL), a low-dose complementarity term in $g_W$ produces an interior optimum (Section 4.3).

In [ ]:
# --- 3.3 + Figure 4: growth-rate response to the waste fraction phi ---
phi=np.linspace(0,0.9,361)
CFq=.04+.30*phi; mu_q=.055*np.clip(1-.55*phi,0,None)*(.82/(1+3.0*CFq+2.2*phi)/.82)
CFb=.10+.25*phi
mu_b=.090*np.clip(1-.25*phi,0,None)*(1+1.6*phi*np.exp(-phi/.12))*(.75/(1+1.0*CFb+.30*phi)/.75)
m_q,m_b=.018,.050
def breakeven(mu,m):
    i=np.where(np.diff(np.sign(mu-m))!=0)[0]
    return np.interp(0,[mu[i[0]]-m,mu[i[0]+1]-m],[phi[i[0]],phi[i[0]+1]]) if len(i) else np.nan
# Monte Carlo band for quail (penalty-parameter uncertainty)
rng=np.random.default_rng(7); MU=np.zeros((600,len(phi)))
for r in range(600):
    a=np.clip(rng.normal(.55,.07),.1,.95); g=np.clip(rng.normal(2.2,.35),.8,4.5)
    MU[r]=.055*np.clip(1-a*phi,0,None)*(.82/(1+3.0*CFq+g*phi)/.82)
lo,med,hi=np.percentile(MU,[5,50,95],axis=0)
fig,axs=plt.subplots(1,2,figsize=(11,4))
axs[0].fill_between(phi,lo,hi,color='#4299e1',alpha=.3,label='5–95% (n=600)')
axs[0].plot(phi,med,color='#2b6cb0',lw=2); axs[0].axhline(m_q,color='#c53030',ls='--')
axs[0].axvline(breakeven(med,m_q),color='k',ls=':'); axs[0].legend(fontsize=8); axs[0].grid(alpha=.3)
axs[0].set_title('(a) Quail: monotonic penalty, φ* = %.2f'%breakeven(med,m_q))
axs[1].plot(phi,mu_b,color='#2f855a',lw=2); axs[1].axhline(m_b,color='#c53030',ls='--')
axs[1].axvline(phi[np.argmax(mu_b)],color='k',ls=':'); axs[1].axvline(breakeven(mu_b,m_b),color='gray',ls=':'); axs[1].grid(alpha=.3)
axs[1].set_title('(b) BSFL: φ_opt = %.2f, φ* = %.2f'%(phi[np.argmax(mu_b)],breakeven(mu_b,m_b)))
for a in axs: a.set_xlabel('waste-derived feed fraction φ'); a.set_ylabel('weekly growth rate')
plt.tight_layout(); plt.show()

### 3.4 Population dynamics

$$N_j(t+1)=N_j(t)\bigl[1+\mu_j(t)-m_j(t)\bigr],\qquad m_j=m_j^{\mathrm{base}}+m_j^{\mathrm{waste}}\phi_j g_m,\qquad \mathbf n^{\mathrm{waste}}_j=N_j\,\mathbf q_j^{\mathrm{waste}}$$

— a closed loop: waste affects growth, growth affects population, population affects waste.

### 3.5 Total causal effects: the nutrient Leontief inverse

With routing fractions $c_{ij}$ and link retentions $\boldsymbol\eta_{i\to j}$, nutrient $k$ satisfies $\mathbf x^{(k)}=A^{(k)}\mathbf x^{(k)}+\mathbf y^{(k)}$ with $A^{(k)}_{ji}=c_{ij}\eta^{(k)}_{i\to j}$, hence

$$T^{(k)}=(I-A^{(k)})^{-1},\qquad T^{(k)}_{ij}=\text{total causal effect of input at }j\text{ on output at }i=\sum_{\text{paths }j\to i}\prod_{e}\eta_e$$

Stability requires spectral radius $\rho(A^{(k)})<1$.

In [ ]:
# --- 3.5 + Figure 3: per-nutrient flow matrix, Leontief inverse, stability ---
nutr=['ME','CP','EE','Ca']; N=6
y=np.zeros((N,4)); y[0]=[5200.,320.,25.,12.]      # weekly external input to plants
export_share=np.array([.08,.10,.15,.25,.05,0.])
T_sol,O_sol={},{}
for k in range(4):
    E=np.zeros((N,N))
    for (i,j),v in eta.items(): E[i,j]=v[k]
    A=(C*E).T
    rho=max(abs(np.linalg.eigvals(A)))
    assert rho<1, 'linear layer unstable'
    T_sol[k]=np.linalg.inv(np.eye(N)-A); O_sol[k]=T_sol[k]@y[:,k]
    print(f'{nutr[k]}: spectral radius = {rho:.2f}  (stable)')
# Figure 3
fig,axs=plt.subplots(2,2,figsize=(11,9),constrained_layout=True)
for k,ax in enumerate(axs.flat):
    Tm=T_sol[k]; L=np.where(Tm>1,np.log10(Tm),0)
    ax.imshow(L,cmap='magma',vmin=0,vmax=L.max())
    ax.set_xticks(range(N)); ax.set_xticklabels(nodes,rotation=30,ha='right',fontsize=8)
    ax.set_yticks(range(N)); ax.set_yticklabels(nodes,fontsize=8)
    ax.set_title(f'nutrient {nutr[k]} ($\\log_{{10}} T_{{ij}}$)')
fig.suptitle('Figure 3 — Total causal effect matrices $T=(I-A)^{-1}$'); plt.show()

### 3.6 Waste quality index — and Section 4.2 results

$$WQ_{j\to k}=\frac{\mathbf w_j\odot\mathbf d_{jk}}{\lVert \mathbf n_k^{\mathrm{req}}\rVert}$$

requirement-normalized, digestibility-discounted nutrient content of a waste stream.

In [ ]:
# --- 3.6 + Figure 2: steady state, indirect-path shares, waste quality index ---
fig,axs=plt.subplots(1,2,figsize=(11.5,4.2)); x=np.arange(N); w=0.2
for k in range(4): axs[0].bar(x+(k-1.5)*w,O_sol[k],width=w,label=nutr[k],color=plt.cm.viridis(k/3))
axs[0].set_yscale('log'); axs[0].set_xticks(x); axs[0].set_xticklabels(nodes,rotation=15,fontsize=9)
axs[0].set_ylabel('weekly throughput (log)'); axs[0].legend(fontsize=8); axs[0].grid(axis='y',alpha=.3)
axs[0].set_title('(a) Steady-state nutrient throughput')
feedcomp={1:[1350,620,90,12],2:[1500,210,140,25],3:[1300,170,60,20]}
d_BSFL=np.array([.35,.55,.50,.40]); req=np.array([1100,170,25,6.])
names=['Cricket frass','Quail manure','Fish sludge']
WQ={nm:(np.array(feedcomp[j])*(1-np.array(eta[(j,4)])))*d_BSFL/req for j,nm in [(1,names[0]),(2,names[1]),(3,names[2])]}
for ii,nm in enumerate(names):
    axs[1].bar(np.arange(4)+(ii-1)*0.26,WQ[nm],width=0.26,label=nm,color=['#4299e1','#48bb78','#ed8936'][ii])
axs[1].axhline(1,color='k',ls='--',lw=1); axs[1].set_xticks(range(4)); axs[1].set_xticklabels(nutr)
axs[1].set_ylim(0,1.6); axs[1].set_ylabel('$WQ_{j\\to BSFL}$'); axs[1].legend(fontsize=8)
axs[1].set_title('(b) Waste quality vs. BSFL requirement')
plt.tight_layout(); plt.show()
# Table: indirect-path shares for CP
k=1; A1=(C*np.array([[eta.get((i,j),[0]*4)[k] for j in range(N)] for i in range(N)])).T
print('CP — direct vs total (indirect share):')
for i in range(N):
    d=(A1@y[:,k])[i]; t=O_sol[k][i]
    print(f'  {nodes[i]:<9} direct {d:7.2f}  total {t:7.2f}  indirect {100*(1-d/t):.1f}%')
print('CP exports/wk:',np.round(export_share*O_sol[k],1))
print('WQI (ME,CP,EE,Ca):',{k:np.round(v,2) for k,v in WQ.items()})

### 3.7–3.9 Closure, stochastic formulation, and the culling governor

Export is only what remains after internal requirements and reserves: $X_p=P_p-D_p-R_p\ge0$, with no drawdown of biological capital. Parameters are distributions (hatch, sex ratio, survival, yields); constraints are probabilistic: $\Pr(\text{failure})\le\alpha$. The maximum safe weekly cull is

$$C^{\max}(t)=\max\Bigl\{C:\Pr\bigl[\min_{0\le k\le L}\bigl(F_{t+k}-F^{\mathrm{req}}_{t+k},\,M_{t+k}-M^{\mathrm{req}}_{t+k}\bigr)<0\bigr]\le\alpha_Q\Bigr\}$$

In [ ]:
# --- 3.9 demo: probabilistic culling governor ---
rng=np.random.default_rng(3)
F_now,L,grow=120,8,1.03                      # hens now; 8-wk horizon; 3%/wk growth target
F_req=lambda k: F_now*(1.05**k)              # required future breeder path (demand-driven)
def fail_prob(C,nsim=20000):
    fails=0
    for _ in range(nsim):
        b=F_now-C; ok=True
        for k in range(1,L+1):
            b=b*(1+grow-1)*rng.normal(1.0,0.04)+rng.poisson(2.0)   # growth + stochastic recruits
            if b<F_req(k): ok=False; break
        fails+=not ok
    return fails/nsim
lo,hi=0,60
for _ in range(12):                          # binary search on C at alpha=1%
    mid=(lo+hi)/2
    if fail_prob(mid)<0.01: lo=mid
    else: hi=mid
print(f'Max safe cull now: {lo:.0f} hens (failure prob at C=0: {fail_prob(0):.3f}, at C={lo:.0f}: {fail_prob(lo):.3f})')

## 4. Discussion, limitations, conclusion

**Findings.** (1) Indirect pathways carry 34–60% of steady-state protein throughput — waste-as-loss models systematically underestimate productivity. (2) The break-even waste fraction $\phi^*$ is experimentally accessible and immediately actionable (keep diets below $\phi^*$; blend waste streams to cover the nutrient gaps the WQI exposes). (3) The Leontief inverse makes the design problem well-posed: maximize export over routing fractions subject to spectral-radius, waste-fraction, and reproductive constraints.

**Limitations.** Parameters are illustrative and must be replaced by measured feed compositions and digestibilities; routing fractions are fixed within a week (adaptive routing is the natural control extension); micro-nutrients, anti-nutritional factors, pathogen dynamics, and founder-population genetics are not yet resolved.

**Conclusion.** Waste is not a loss term; it is a measurable, priceable, growth-modulating input. Expressed as a causal network, a circular food system becomes analyzable, falsifiable, and directly optimizable.

## Appendix — weekly simulation algorithm & skeleton

1. Observe cohorts, sex counts, inventories. 2. Realize egg production; allocate to food/export/incubation/reserve. 3. Realize hatch and sexes. 4. Age cohorts; apply survival. 5. Compute $\phi_j,\mu_j$; update populations. 6. Generate wastes; assemble feeds. 7. Project breeders over horizon $L$; compute $C^{\max}(t)$; execute $C=\min[C^{\mathrm{demand}},C^{\max}]$. 8. Check closure, water, energy; compute exportable surplus. 9. Record constraint margins; advance.

In [ ]:
# --- Appendix: reusable simulation components ---
class Organism:
    def __init__(self, name, d, r):
        self.name=name; self.d=np.asarray(d,float); self.r=np.asarray(r,float)
    def feed(self, f):
        f=np.asarray(f,float)
        digested=f*self.d; retained=digested*self.r
        return f*(1-self.d), digested*(1-self.r), retained          # waste, product, retained

class BSFLFarm:
    def __init__(self, d, c): self.d=np.asarray(d,float); self.c=c
    def process(self, waste_stream):
        return waste_stream*self.d*self.c, waste_stream*(1-self.d)  # larvae, frass

# example: 1 kg of plant-mix biomass through cricket -> BSFL
plant=np.array([1500,110,15,4.]); cricket=Organism('cricket',[.50,.55,.40,.10],[.6,.65,.55,.15])
bsfl=BSFLFarm([.35,.55,.50,.40],.45)
w_cr,p_cr,ret_cr=cricket.feed(plant)
larvae,frass=bsfl.process(w_cr)
print('cricket waste -> BSFL larvae:',np.round(larvae,1),'| frass to soil:',np.round(frass,1))

## References

1. Leontief (1941), *The Structure of American Economy*.
2. Tilley & Terry (1963), *J. Br. Grassl. Soc.* 18:104.
3. Gompertz (1825), *Phil. Trans. R. Soc.* 115:513.
4. Lotka (1925), *Elements of Physical Biology*.
5. Caswell (2001), *Matrix Population Models*, 2nd ed.
6. van Huis et al. (2013), FAO Forestry Paper 171.
7. Makkar et al. (2014), *Anim. Feed Sci. Technol.* 197:1.
8. Lundy & Parrella (2015), *PLoS ONE* 10(4):e0118785.
9. Diener et al. (2009), *Waste Manage. Res.* 27:603.
10. Sheppard et al. (1994), *Bioresour. Technol.* 50:275.
11. Henry et al. (2015), *J. Anim. Sci. Biotechnol.* 6:12.
12. Lalander et al. (2019), *J. Clean. Prod.* 208:211.

*Verify page/volume details before camera-ready submission.*